# NLP&TM Assignment 

Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report


Load the Data

In [3]:
df = pd.read_csv("quora.csv")

In [5]:
df

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0
...,...,...,...,...,...,...
404285,404285,433578,379845,How many keywords are there in the Racket prog...,How many keywords are there in PERL Programmin...,0
404286,404286,18840,155606,Do you believe there is life after death?,Is it true that there is life after death?,1
404287,404287,537928,537929,What is one coin?,What's this coin?,0
404288,404288,537930,537931,What is the approx annual cost of living while...,I am having little hairfall problem but I want...,0


In [ ]:
text preprocessing function

In [7]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    return ' '.join(words)

# Δημιουργούμε τις καθαρές στήλες
df['q1_clean'] = df['question1'].apply(clean_text)
df['q2_clean'] = df['question2'].apply(clean_text)


This block defines a text preprocessing function that converts text to lowercase, removes punctuation, filters out English stopwords, and applies stemming using the Porter Stemmer. The function is then applied to both questions in the dataset, creating two new cleaned columns: q1_clean and q2_clean.

Set up stopwords and stemmer

In [9]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [ ]:
defines two functions to compute text similarity

In [11]:
def get_common_words(q1, q2):
    return len(set(q1.split()) & set(q2.split()))

def jaccard_similarity(q1, q2):
    w1 = set(q1.split())
    w2 = set(q2.split())
    if len(w1 | w2) == 0:
        return 0
    return len(w1 & w2) / len(w1 | w2)

df['common_words'] = df.apply(lambda row: get_common_words(row['q1_clean'], row['q2_clean']), axis=1)
df['jaccard'] = df.apply(lambda row: jaccard_similarity(row['q1_clean'], row['q2_clean']), axis=1)
df['len_diff'] = abs(df['q1_clean'].str.len() - df['q2_clean'].str.len())
df['word_count_diff'] = abs(df['q1_clean'].str.split().str.len() - df['q2_clean'].str.split().str.len())

# Δείχνουμε παραδείγματα
df[['q1_clean', 'q2_clean', 'common_words', 'jaccard', 'len_diff', 'word_count_diff']].head()


,q1_clean,q2_clean,common_words,jaccard,len_diff,word_count_diff
0,step step guid invest share market india,step step guid invest share market,5,0.833333,6,1
1,stori kohinoor kohinoor diamond,would happen indian govern stole kohinoor kohi...,2,0.222222,32,5
2,increas speed internet connect use vpn,internet speed increas hack dn,3,0.375000,8,1
3,mental lone solv,find remaind math2324math divid 2423,0,0.000000,20,2
4,one dissolv water quikli sugar salt methan car...,fish would surviv salt water,2,0.153846,29,5


This code defines two functions to compute text similarity: one that counts the number of common words between two questions, and another that calculates their Jaccard similarity. Then, it creates four new features in the dataset—common_words, jaccard, len_diff, and word_count_diff—which quantify how similar or different each pair of cleaned questions is. Finally, it displays the first few rows of these new features.

 Feature Extraction (TF-IDF + Cosine Similarity)

In [13]:
# Επιλέγουμε τα νέα features για το μοντέλο
features = ['common_words', 'jaccard', 'len_diff', 'word_count_diff']
X = df[features]
y = df['is_duplicate']

# train/test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Random Forest
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Προβλέπουμε στο test set
y_pred = clf.predict(X_test)

# Μετρήσεις απόδοσης
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7020538062611822
F1 Score: 0.5628183258931272

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.81      0.77     76350
           1       0.62      0.52      0.56     44937

    accuracy                           0.70    121287
   macro avg       0.68      0.66      0.67    121287
weighted avg       0.69      0.70      0.70    121287



This part of the code selects four engineered features to train a machine learning model that predicts whether question pairs are duplicates. It splits the dataset into training and test sets, trains a Random Forest classifier, and makes predictions on the test set. Finally, it evaluates the model’s performance using accuracy, F1 score, and a classification report.

In [15]:
df.head(10).style.set_properties(**{'background-color': '#f0f8ff',
                                    'color': 'black',
                                    'border-color': 'gray'})

,id,qid1,qid2,question1,question2,is_duplicate,q1_clean,q2_clean,common_words,jaccard,len_diff,word_count_diff
0,0,1,2,What is the step by step guide to invest in share market in india?,What is the step by step guide to invest in share market?,0,step step guid invest share market india,step step guid invest share market,5,0.833333,6,1
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Diamond?,What would happen if the Indian government stole the Kohinoor (Koh-i-Noor) diamond back?,0,stori kohinoor kohinoor diamond,would happen indian govern stole kohinoor kohinoor diamond back,2,0.222222,32,5
2,2,5,6,How can I increase the speed of my internet connection while using a VPN?,How can Internet speed be increased by hacking through DNS?,0,increas speed internet connect use vpn,internet speed increas hack dn,3,0.375000,8,1
3,3,7,8,Why am I mentally very lonely? How can I solve it?,"Find the remainder when [math]23^{24}[/math] is divided by 24,23?",0,mental lone solv,find remaind math2324math divid 2423,0,0.000000,20,2
4,4,9,10,"Which one dissolve in water quikly sugar, salt, methane and carbon di oxide?",Which fish would survive in salt water?,0,one dissolv water quikli sugar salt methan carbon di oxid,fish would surviv salt water,2,0.153846,29,5
5,5,11,12,Astrology: I am a Capricorn Sun Cap moon and cap rising...what does that say about me?,"I'm a triple Capricorn (Sun, Moon and ascendant in Capricorn) What does this say about me?",1,astrolog capricorn sun cap moon cap risingwhat say,im tripl capricorn sun moon ascend capricorn say,4,0.400000,2,0
6,6,13,14,Should I buy tiago?,What keeps childern active and far from phone and video games?,0,buy tiago,keep childern activ far phone video game,0,0.000000,31,5
7,7,15,16,How can I be a good geologist?,What should I do to be a great geologist?,1,good geologist,great geologist,1,0.333333,1,0
8,8,17,18,When do you use シ instead of し?,"When do you use ""&"" instead of ""and""?",0,use instead,use instead,2,1.000000,0,0
9,9,19,20,Motorola (company): Can I hack my Charter Motorolla DCX3400?,How do I hack Motorola DCX3400 for free internet?,0,motorola compani hack charter motorolla dcx3400,hack motorola dcx3400 free internet,3,0.375000,12,1
